# 03 — OCR: Per-Digit Classifier

Train a ResNet18 digit classifier on UM individual digit crops.
Evaluate on WM polygon (grid-split of warped ROI) and WM bbox (grid-split of bbox crop).

In [ ]:
import sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if not IN_COLAB:
    try:
        from google.colab import drive
        IN_COLAB = True
    except ImportError:
        pass

if IN_COLAB:
    from google.colab import drive, userdata
    drive.mount("/content/drive")
    token = userdata.get("GITHUB_TOKEN", "")
    base = f"https://{token}@github.com" if token else "https://github.com"
    if not Path("/content/WaterMeterCV").exists():
        subprocess.run(["git", "clone", f"{base}/UrranQx/WaterMeterCV.git", "/content/WaterMeterCV"], check=True)
    BRANCH = "feature/ocr-notebooks"
    subprocess.run(["git", "-C", "/content/WaterMeterCV", "checkout", BRANCH], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "rapidfuzz", "shapely"], check=True)
    ROOT         = Path("/content/WaterMeterCV")
    DATA_ROOT    = Path("/content/drive/MyDrive/WaterMeterCV/WaterMetricsDATA")
    WEIGHTS_BASE = Path("/content/drive/MyDrive/WaterMeterCV/weights")
    RESULTS_DIR  = Path("/content/drive/MyDrive/WaterMeterCV/results")
    WORKERS = 2
else:
    ROOT         = Path("../..")
    DATA_ROOT    = ROOT / "WaterMetricsDATA"
    WEIGHTS_BASE = ROOT / "models/weights"
    RESULTS_DIR  = ROOT / "results"
    WORKERS = 0

sys.path.insert(0, str(ROOT))
WEIGHTS_BASE.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import json, time, csv
import torch, cv2, numpy as np
from datetime import datetime

print(f"ROOT: {ROOT}")
print(f"torch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
CROPS_ROOT   = DATA_ROOT / "ocr_crops"
UM_YOLO_PATH = DATA_ROOT / "utility-meter-reading-dataset-for-automatic-reading-yolo.v4i.yolov11"
WM_PATH      = DATA_ROOT / "waterMeterDataset/WaterMeters"
WEIGHTS_DIR  = WEIGHTS_BASE / "ocr_per_digit"
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)

DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CROP_SIZE  = 32
EPOCHS     = 15
BATCH_SIZE = 64
LR         = 1e-3
RUN_NAME   = f"perdigit_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
print(f"Device: {DEVICE}")
print(f"Run: {RUN_NAME}")

In [ ]:
from models.data.ocr_dataset import (
    prepare_ocr_crops, load_ocr_crops, load_um_digit_crops,
    warp_roi_polygon, crop_roi_bbox, CHARSET,
)
from models.data.unified_loader import load_water_meter_dataset_split
from models.metrics.evaluation import full_string_accuracy, per_digit_accuracy, character_error_rate

print("Preparing WM OCR crops (skips existing)...")
prepare_ocr_crops(WM_PATH, CROPS_ROOT)
print("Done.")

In [ ]:
import torch.nn as nn
import torchvision.models as tv_models
import torchvision.transforms.functional as TF
from torch.utils.data import TensorDataset, DataLoader
from tqdm import tqdm


def build_digit_classifier():
    m = tv_models.resnet18(weights=tv_models.ResNet18_Weights.DEFAULT)
    m.fc = nn.Linear(m.fc.in_features, 10)
    return m


def train_classifier(model, loader, epochs, device):
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    crit = nn.CrossEntropyLoss()
    for epoch in range(epochs):
        model.train()
        total, correct = 0, 0
        for x, y in tqdm(loader, desc=f"epoch {epoch+1}/{epochs}", leave=False):
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss = crit(out, y)
            opt.zero_grad(); loss.backward(); opt.step()
            correct += (out.argmax(1) == y).sum().item()
            total   += len(y)
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"  epoch {epoch+1}/{epochs}  acc={correct/total:.3f}")
    return model


@torch.no_grad()
def classify_crops(model, crops_bgr: list, device) -> list[int]:
    model.eval()
    tensors = torch.stack([
        TF.to_tensor(cv2.cvtColor(cv2.resize(c, (CROP_SIZE, CROP_SIZE)), cv2.COLOR_BGR2RGB))
        for c in crops_bgr
    ]).to(device)
    return model(tensors).argmax(1).tolist()


def orient_fix(pred: str) -> str:
    rev = pred[::-1]
    return rev if rev.isdigit() else pred


print("Helpers loaded.")

In [ ]:
# Train on UM individual digit crops (UM has digit-level GT bboxes)
raw_crops = load_um_digit_crops(UM_YOLO_PATH, "train")
print(f"UM digit crops: {len(raw_crops)}")
imgs   = torch.stack([TF.to_tensor(cv2.cvtColor(c, cv2.COLOR_BGR2RGB)) for c, _ in raw_crops])
labels = torch.tensor([cls for _, cls in raw_crops], dtype=torch.long)
loader = DataLoader(TensorDataset(imgs, labels), batch_size=BATCH_SIZE, shuffle=True, num_workers=WORKERS)

model = build_digit_classifier()
print(f"Training ({EPOCHS} epochs)...")
model = train_classifier(model, loader, EPOCHS, DEVICE)
torch.save(model.state_dict(), WEIGHTS_DIR / f"best_{RUN_NAME}.pth")
print("Training done.")

In [ ]:
def eval_wm_grid(model, crop_key, device):
    wm_train_raw, wm_test_raw = load_water_meter_dataset_split(WM_PATH, train_ratio=0.7, seed=42)
    wm_test = [s for s in wm_test_raw if s.roi_polygon is not None and s.value is not None]
    preds, gts, times = [], [], []
    for s in wm_test:
        img = cv2.imread(str(s.image_path))
        if img is None:
            continue
        label_gt = str(int(s.value))
        n = len(label_gt)
        t0 = time.perf_counter()
        if crop_key == "wm_polygon":
            roi_crop = warp_roi_polygon(img, s.roi_polygon, out_h=CROP_SIZE, out_w=CROP_SIZE * n)
        else:
            from models.data.roi_dataset import polygon_to_bbox
            roi_crop = crop_roi_bbox(img, polygon_to_bbox(s.roi_polygon), out_h=CROP_SIZE, out_w=CROP_SIZE * n)
        col_w = roi_crop.shape[1] // n
        cols  = [roi_crop[:, i*col_w:(i+1)*col_w] for i in range(n)]
        preds_cls = classify_crops(model, cols, device)
        times.append((time.perf_counter() - t0) * 1000)
        pred = orient_fix("".join(str(c) for c in preds_cls))
        preds.append(pred)
        gts.append(label_gt)
    fsa_raw  = full_string_accuracy(preds, gts)
    fsa_norm = full_string_accuracy([p.lstrip("0") or "0" for p in preds],
                                    [g.lstrip("0") or "0" for g in gts])
    pda = float(np.mean([per_digit_accuracy(p, g) for p, g in zip(preds, gts)]))
    cer = float(np.mean([character_error_rate(p, g) for p, g in zip(preds, gts)]))
    ms  = float(np.mean(times))
    return fsa_raw, fsa_norm, pda, cer, ms, preds, gts


wm_poly_fsa_raw, wm_poly_fsa_norm, wm_poly_pda, wm_poly_cer, wm_poly_ms, wm_poly_preds, wm_poly_gts = eval_wm_grid(model, "wm_polygon", DEVICE)
print(f"WM polygon — FSA raw={wm_poly_fsa_raw:.3f} norm={wm_poly_fsa_norm:.3f} PDA={wm_poly_pda:.3f} CER={wm_poly_cer:.3f} {wm_poly_ms:.1f}ms")

wm_bbox_fsa_raw, wm_bbox_fsa_norm, wm_bbox_pda, wm_bbox_cer, wm_bbox_ms, wm_bbox_preds, wm_bbox_gts = eval_wm_grid(model, "wm_bbox", DEVICE)
print(f"WM bbox    — FSA raw={wm_bbox_fsa_raw:.3f} norm={wm_bbox_fsa_norm:.3f} PDA={wm_bbox_pda:.3f} CER={wm_bbox_cer:.3f} {wm_bbox_ms:.1f}ms")

In [ ]:
print(f"{'='*55}")
print(f"{'Metric':<20} {'WM polygon':>15} {'WM bbox':>15}")
print(f"{'='*55}")
for name, poly_v, bbox_v in [
    ("FSA raw",   wm_poly_fsa_raw,  wm_bbox_fsa_raw),
    ("FSA norm",  wm_poly_fsa_norm, wm_bbox_fsa_norm),
    ("Per-digit", wm_poly_pda,      wm_bbox_pda),
    ("CER",       wm_poly_cer,      wm_bbox_cer),
]:
    print(f"{name:<20} {poly_v:>15.3f} {bbox_v:>15.3f}")
print(f"{'Inference ms':<20} {wm_poly_ms:>15.1f} {wm_bbox_ms:>15.1f}")
print(f"{'='*55}")

In [ ]:
# Visualization: 2x4 grid — WM polygon row, WM bbox row
fig, axes = plt.subplots(2, 4, figsize=(20, 8))

wm_poly_crops = load_ocr_crops(CROPS_ROOT / "wm_polygon", "test")
for ax_idx, ax in enumerate(axes[0]):
    if ax_idx < len(wm_poly_crops):
        img = cv2.imread(str(wm_poly_crops[ax_idx][0]))
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        gt = wm_poly_gts[ax_idx] if ax_idx < len(wm_poly_gts) else "?"
        pr = wm_poly_preds[ax_idx] if ax_idx < len(wm_poly_preds) else "?"
        ax.set_title(f"GT={gt} Pred={pr}", fontsize=9)
    ax.axis("off")

wm_bbox_crops = load_ocr_crops(CROPS_ROOT / "wm_bbox", "test")
for ax_idx, ax in enumerate(axes[1]):
    if ax_idx < len(wm_bbox_crops):
        img = cv2.imread(str(wm_bbox_crops[ax_idx][0]))
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        gt = wm_bbox_gts[ax_idx] if ax_idx < len(wm_bbox_gts) else "?"
        pr = wm_bbox_preds[ax_idx] if ax_idx < len(wm_bbox_preds) else "?"
        ax.set_title(f"GT={gt} Pred={pr}", fontsize=9)
    ax.axis("off")

axes[0][0].set_ylabel("WM polygon", fontsize=11)
axes[1][0].set_ylabel("WM bbox", fontsize=11)
plt.suptitle("Per-Digit Classifier Predictions", fontsize=14)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "ocr_per_digit_predictions.png", dpi=150)
plt.close()
print("Saved ocr_per_digit_predictions.png")

In [ ]:
metrics = {
    "method": "per_digit_classifier",
    "wm_polygon":     {"fsa_raw": round(wm_poly_fsa_raw,4), "fsa_norm": round(wm_poly_fsa_norm,4),
                       "per_digit_acc": round(wm_poly_pda,4), "cer": round(wm_poly_cer,4),
                       "avg_inference_ms": round(wm_poly_ms,1)},
    "wm_bbox":        {"fsa_raw": round(wm_bbox_fsa_raw,4), "fsa_norm": round(wm_bbox_fsa_norm,4),
                       "per_digit_acc": round(wm_bbox_pda,4), "cer": round(wm_bbox_cer,4),
                       "avg_inference_ms": round(wm_bbox_ms,1)},
    "config": {"epochs": EPOCHS, "batch_size": BATCH_SIZE, "lr": LR, "backbone": "resnet18"},
    "run_date": datetime.now().isoformat(),
}
with open(RESULTS_DIR / "ocr_per_digit_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

csv_path = RESULTS_DIR / "ocr_comparison.csv"
header = ["method",
          "wm_poly_fsa_raw","wm_poly_fsa_norm","wm_poly_pda","wm_poly_cer","wm_poly_ms",
          "wm_bbox_fsa_raw","wm_bbox_fsa_norm","wm_bbox_pda","wm_bbox_cer","wm_bbox_ms","run_date"]
write_hdr = not csv_path.exists()
with open(csv_path, "a", newline="") as f:
    w = csv.writer(f)
    if write_hdr: w.writerow(header)
    w.writerow(["per_digit_classifier",
                round(wm_poly_fsa_raw,4), round(wm_poly_fsa_norm,4), round(wm_poly_pda,4), round(wm_poly_cer,4), round(wm_poly_ms,1),
                round(wm_bbox_fsa_raw,4), round(wm_bbox_fsa_norm,4), round(wm_bbox_pda,4), round(wm_bbox_cer,4), round(wm_bbox_ms,1),
                datetime.now().strftime("%Y-%m-%d %H:%M")])
print("Results saved.")

## Conclusions

- **WM polygon:** FSA raw=?, FSA norm=?, PDA=?, CER=?, ?ms
- **WM bbox:** FSA raw=?, FSA norm=?, PDA=?, CER=?, ?ms
- **Key finding:** fill after running.
- **Grid-split limitation:** splits ROI into N equal columns assuming uniform digit width — performance depends on meter layout regularity.